# 12 — AG-UI Streaming

**AWS-side equivalent:** AG-UI 16-event taxonomy over SSE on AgentCore Runtime
**GCP service:** Same AG-UI taxonomy, self-hosted on Cloud Run (or Agent Engine for production)

AWS-side notebook: `notebooks/aws/12-ag-ui-streaming.ipynb`

_As of 2026-05: there is no GCP-managed AG-UI server. The protocol layer is the same `ag-ui-protocol` Python package (`ag_ui.core` for event models, `ag_ui.encoder` for SSE wire format). We host it ourselves with FastAPI + a Gemini call; deploy the same handler to Cloud Run when promoting beyond the notebook._


## Steps

1. Build a tiny Gemini-backed handler that yields the AG-UI event sequence (RUN_STARTED → TEXT_MESSAGE_* → TOOL_CALL_* → RUN_FINISHED)
2. Wrap it in FastAPI as a `StreamingResponse(media_type="text/event-stream")`
3. Run uvicorn in a background thread inside the notebook
4. Hit it with a POST + ReadableStream-style client (`httpx.stream`) and print the decoded events


## Step 1: Load config


In [ ]:
import json
from pathlib import Path

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
MODEL_ID = gcp.get('model_id', 'gemini-2.5-flash')  # flash for cheaper streaming demo
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
print(f'project={PROJECT_ID} region={REGION} model={MODEL_ID}')


## Step 2: Define the AG-UI event generator

`ag_ui.core` provides 16 Pydantic event classes; we use 7 of them here. Each event carries a `type` enum drawn from `EventType`. `EventEncoder.encode(event)` returns the SSE wire chunk (`data: {...}\n\n`).

The generator below simulates the same lifecycle a real ADK / Gemini stream would emit: start → tool call (a stub `search_experts` lookup) → streamed model text → finish. Switching to a real Gemini stream is a one-line change (`client.models.generate_content_stream(...)` → yield `TextMessageContentEvent` per chunk); we keep the canned sequence here so the demo runs offline and is fully deterministic.


In [ ]:
import uuid
from typing import AsyncIterator

from ag_ui.core import (
    EventType,
    RunStartedEvent,
    RunFinishedEvent,
    TextMessageStartEvent,
    TextMessageContentEvent,
    TextMessageEndEvent,
    ToolCallStartEvent,
    ToolCallArgsEvent,
    ToolCallEndEvent,
)
from ag_ui.encoder import EventEncoder


async def orgiq_event_stream(thread_id: str, run_id: str, user_message: str) -> AsyncIterator[object]:
    """Yield the AG-UI event sequence for a single OrgIQ-search style request."""
    msg_id = f'msg_{uuid.uuid4().hex[:8]}'
    tool_id = f'tool_{uuid.uuid4().hex[:8]}'

    yield RunStartedEvent(type=EventType.RUN_STARTED, thread_id=thread_id, run_id=run_id)

    # 1. The agent decides to call a tool.
    yield ToolCallStartEvent(
        type=EventType.TOOL_CALL_START,
        tool_call_id=tool_id,
        tool_call_name='search_experts',
        parent_message_id=msg_id,
    )
    yield ToolCallArgsEvent(
        type=EventType.TOOL_CALL_ARGS,
        tool_call_id=tool_id,
        delta=json.dumps({'skill': 'kubernetes', 'limit': 3}),
    )
    yield ToolCallEndEvent(type=EventType.TOOL_CALL_END, tool_call_id=tool_id)

    # 2. The agent streams its narrative answer.
    yield TextMessageStartEvent(type=EventType.TEXT_MESSAGE_START, message_id=msg_id, role='assistant')
    answer = (
        f'Based on the org graph, three engineers stand out for "{user_message}": '
        'Alice Chen, Bob Kumar, and Carol Patel. '
        'Alice has the highest composite expertise score (0.91).'
    )
    # Chunk the answer to demonstrate streaming.
    chunk = 32
    for i in range(0, len(answer), chunk):
        yield TextMessageContentEvent(
            type=EventType.TEXT_MESSAGE_CONTENT,
            message_id=msg_id,
            delta=answer[i:i + chunk],
        )
    yield TextMessageEndEvent(type=EventType.TEXT_MESSAGE_END, message_id=msg_id)

    yield RunFinishedEvent(type=EventType.RUN_FINISHED, thread_id=thread_id, run_id=run_id)

print('✓ orgiq_event_stream generator defined')


## Step 3: Encode to SSE — smoke-test the generator

Run the generator without HTTP first so we can see the wire format and confirm event ordering before the FastAPI hop.


In [ ]:
encoder = EventEncoder()

async def _smoke_test():
    captured = []
    async for ev in orgiq_event_stream('thread-001', 'run-001', 'kubernetes experts'):
        captured.append(encoder.encode(ev))
    return captured

events = await _smoke_test()
print(f'✓ emitted {len(events)} SSE chunks\n')
for i, chunk in enumerate(events[:6]):
    print(f'--- chunk {i} ---')
    print(chunk.rstrip())
print(f'\n…({len(events) - 6} more chunks)')


## Step 4: Wrap the generator in a FastAPI handler

AG-UI uses **POST** for SSE (not the GET-based EventSource), so the body is a `RunAgentInput` JSON. The handler returns a `StreamingResponse` with `media_type='text/event-stream'`. The exact same handler ships to Cloud Run unchanged.


In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse

app = FastAPI(title='OrgIQ AG-UI demo')


@app.post('/agent/run')
async def agent_run(request: Request):
    body = await request.json()
    thread_id = body.get('thread_id', f'thread_{uuid.uuid4().hex[:8]}')
    run_id = body.get('run_id', f'run_{uuid.uuid4().hex[:8]}')
    messages = body.get('messages', [])
    user_msg = messages[-1]['content'] if messages else 'who knows kubernetes?'

    enc = EventEncoder()

    async def gen():
        async for ev in orgiq_event_stream(thread_id, run_id, user_msg):
            yield enc.encode(ev)

    return StreamingResponse(gen(), media_type='text/event-stream')


print('✓ FastAPI app defined; route POST /agent/run')


## Step 5: Boot uvicorn in a background thread

Notebook environments don't have an event loop free to host a long-running server. We run uvicorn in a thread on `127.0.0.1:8765` so subsequent cells can hit it like a real client.


In [ ]:
import threading, time
import uvicorn

_server: uvicorn.Server | None = globals().get('_server')

if _server is None or not _server.started:
    config_uv = uvicorn.Config(app, host='127.0.0.1', port=8765, log_level='warning')
    _server = uvicorn.Server(config_uv)
    threading.Thread(target=_server.run, daemon=True).start()
    # Bounded wait for the server to bind (cap at ~3s).
    for _ in range(30):
        if _server.started:
            break
        time.sleep(0.1)

print(f'✓ uvicorn started={_server.started} on http://127.0.0.1:8765')


## Step 6: Stream from the client side

`httpx.stream` is the Python analog of `fetch(...).body.getReader()` — it streams the SSE response incrementally so we can render events as they arrive. Each `data: {...}` line round-trips into a JSON object whose `type` field matches the `EventType` enum.


In [ ]:
import httpx

received: list[dict] = []
with httpx.stream(
    'POST',
    'http://127.0.0.1:8765/agent/run',
    json={
        'thread_id': 'thread-demo',
        'run_id': 'run-demo',
        'messages': [{'role': 'user', 'content': 'who knows kubernetes?'}],
    },
    timeout=30,
) as r:
    r.raise_for_status()
    for line in r.iter_lines():
        if line.startswith('data:'):
            payload = json.loads(line[len('data:'):].strip())
            received.append(payload)
            print(f'{payload["type"]:24s} {json.dumps({k: v for k, v in payload.items() if k != "type"})[:80]}')

print(f'\n✓ received {len(received)} events')


## Step 7: Reconstruct the final assistant text

Mirroring the AWS notebook's "render to UI" step: concatenate `TEXT_MESSAGE_CONTENT` deltas keyed by `messageId` to reproduce the visible assistant turn.


In [ ]:
buffers: dict[str, str] = {}
for ev in received:
    if ev['type'] == 'TEXT_MESSAGE_CONTENT':
        mid = ev.get('messageId') or ev.get('message_id')
        buffers[mid] = buffers.get(mid, '') + ev.get('delta', '')

for mid, txt in buffers.items():
    print(f'\n[{mid}]')
    print(txt)

print(f'\n✓ reconstructed {len(buffers)} assistant message(s)')


## Step 8: Persist endpoint info

Once the agent moves to Cloud Run, the only field that changes is `endpoint_url`. The middleware uses the same SSE contract regardless of where it points.


In [ ]:
gcp['ag_ui'] = {
    'endpoint_url': 'http://127.0.0.1:8765/agent/run',
    'model_id': MODEL_ID,
    'events_emitted': sorted({ev['type'] for ev in received}),
    'transport': 'POST text/event-stream',
}
config_path.write_text(json.dumps(config, indent=2))
print('✓ config.json updated under gcp.ag_ui')


## Promoting to Cloud Run

When you're ready to move beyond the notebook:

```bash
# Dockerfile entrypoint: uvicorn app:app --host 0.0.0.0 --port 8080
gcloud run deploy orgiq-ag-ui \
    --source . \
    --region $REGION \
    --allow-unauthenticated=false \
    --set-env-vars=GOOGLE_CLOUD_PROJECT=$PROJECT_ID
```

Cloud Run handles SSE natively (no extra config). For tighter latency budgets, Agent Engine offers a hosted runtime that already speaks AG-UI; use it once the agent stabilises.


## Done

Next: `13-a2a-protocol.ipynb` (agent-to-agent messaging, mirroring the AWS A2A notebook).
